# 05. 비즈니스 임팩트 종합 분석

이전 노트북의 분석 결과를 비즈니스 관점에서 통합하고, 구체적인 ROI와 액션 플랜을 제시합니다.

**분석 내용**:
1. 이탈 비용 추정 (매출 손실 규모)
2. 모델 기반 리텐션 캠페인 ROI 계산
3. 고객 생애가치(CLV) by Segment
4. RFM 세그먼트 × 이탈 위험 매트릭스
5. 프로젝트 3개 축 연결 (FRE + SQL + ML)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## 1. 이탈 비용 추정

Telco 데이터 기반 이탈 비용을 계산합니다.

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

# 이탈 고객 통계
churned = df[df['Churn_Binary'] == 1]
retained = df[df['Churn_Binary'] == 0]

monthly_revenue_at_risk = churned['MonthlyCharges'].sum()
annual_revenue_at_risk = monthly_revenue_at_risk * 12
avg_monthly_charge = churned['MonthlyCharges'].mean()

print('=== 이탈 비용 추정 ===')
print(f'이탈 고객 수: {len(churned):,}')
print(f'이탈 고객 평균 월 요금: ${avg_monthly_charge:.2f}')
print(f'월간 손실 매출: ${monthly_revenue_at_risk:,.0f}')
print(f'연간 손실 매출 (추정): ${annual_revenue_at_risk:,.0f}')
print(f'\n전체 매출 대비 손실: {monthly_revenue_at_risk / df["MonthlyCharges"].sum() * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: 이탈 비용 구조
categories = ['Churned\nMonthly Revenue', 'Retained\nMonthly Revenue']
values = [monthly_revenue_at_risk, retained['MonthlyCharges'].sum()]
colors = ['#e15759', '#59a14f']
bars = axes[0].bar(categories, values, color=colors, width=0.5)
for bar, v in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'${v:,.0f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Monthly Revenue ($)')
axes[0].set_title('Revenue at Risk from Churn', fontsize=14, fontweight='bold')

# Panel 2: Contract별 이탈 매출
contract_churn_rev = churned.groupby('Contract')['MonthlyCharges'].sum().sort_values(ascending=False)
colors_contract = ['#e15759', '#f28e2b', '#4e79a7']
bars = axes[1].bar(contract_churn_rev.index, contract_churn_rev.values, color=colors_contract)
for bar, v in zip(bars, contract_churn_rev.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'${v:,.0f}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Monthly Revenue at Risk ($)')
axes[1].set_title('Churn Revenue by Contract Type', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. 모델 기반 리텐션 캠페인 ROI

ML 모델을 실제 운영에 적용했을 때의 비즈니스 임팩트를 시뮬레이션합니다.

In [ ]:
# 시나리오 설정 (보수적 추정)
MODEL_RECALL = 0.80  # 모델이 이탈자 80% 감지
CAMPAIGN_SUCCESS_RATE = 0.25  # 감지된 이탈자 중 25%가 리텐션 성공
COST_PER_INTERVENTION = 30  # 고객당 리텐션 캠페인 비용 ($30)
AVG_CUSTOMER_LIFETIME_MONTHS = 12  # 리텐션 성공 시 추가 유지 기간

total_churners = len(churned)
detected = int(total_churners * MODEL_RECALL)
saved = int(detected * CAMPAIGN_SUCCESS_RATE)

campaign_cost = detected * COST_PER_INTERVENTION
saved_revenue = saved * avg_monthly_charge * AVG_CUSTOMER_LIFETIME_MONTHS
net_benefit = saved_revenue - campaign_cost
roi = (net_benefit / campaign_cost) * 100

print('=== Retention Campaign ROI Simulation ===')
print(f'\n[Input]')
print(f'  Total churners: {total_churners:,}')
print(f'  Model recall: {MODEL_RECALL:.0%}')
print(f'  Campaign success rate: {CAMPAIGN_SUCCESS_RATE:.0%}')
print(f'  Cost per intervention: ${COST_PER_INTERVENTION}')
print(f'  Avg monthly charge: ${avg_monthly_charge:.2f}')
print(f'\n[Output]')
print(f'  Detected at-risk: {detected:,}')
print(f'  Successfully retained: {saved:,}')
print(f'  Campaign cost: ${campaign_cost:,.0f}')
print(f'  Saved revenue (12mo): ${saved_revenue:,.0f}')
print(f'  Net benefit: ${net_benefit:,.0f}')
print(f'  ROI: {roi:.0f}%')

In [ ]:
# ROI 워터폴 차트
fig, ax = plt.subplots(figsize=(12, 6))

steps = ['Campaign\nCost', 'Saved Revenue\n(12 months)', 'Net Benefit']
values = [-campaign_cost, saved_revenue, net_benefit]
colors = ['#e15759', '#59a14f', '#4e79a7']

bars = ax.bar(steps, values, color=colors, width=0.5)
for bar, v in zip(bars, values):
    offset = 2000 if v > 0 else -4000
    ax.text(bar.get_x() + bar.get_width()/2, v + offset,
            f'${abs(v):,.0f}', ha='center', fontsize=12, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Amount ($)')
ax.set_title(f'Retention Campaign ROI: {roi:.0f}%', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. 고객 생애가치 (CLV) by Segment

In [ ]:
# Contract 기반 세그먼트별 CLV 추정
# CLV = Average Monthly Charges × Average Tenure
segments = df.groupby('Contract').agg(
    Customers=('customerID', 'count'),
    Avg_Monthly=('MonthlyCharges', 'mean'),
    Avg_Tenure=('tenure', 'mean'),
    Churn_Rate=('Churn_Binary', 'mean'),
    Total_Revenue=('TotalCharges', 'sum')
).round(2)

segments['CLV'] = (segments['Avg_Monthly'] * segments['Avg_Tenure']).round(0)
segments['Revenue_Share'] = (segments['Total_Revenue'] / segments['Total_Revenue'].sum() * 100).round(1)

print('=== Customer Lifetime Value by Contract ===')
segments

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['#e15759', '#f28e2b', '#59a14f']

# CLV by Contract
bars = axes[0].bar(segments.index, segments['CLV'], color=colors)
for bar, v in zip(bars, segments['CLV']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'${v:,.0f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('CLV ($)')
axes[0].set_title('Customer Lifetime Value', fontsize=13, fontweight='bold')

# Churn Rate by Contract
bars = axes[1].bar(segments.index, segments['Churn_Rate'] * 100, color=colors)
for bar, v in zip(bars, segments['Churn_Rate'] * 100):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Contract', fontsize=13, fontweight='bold')

# Revenue Share
axes[2].pie(segments['Revenue_Share'], labels=segments.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 10})
axes[2].set_title('Revenue Share by Contract', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. RFM 세그먼트 × 이탈 위험 매트릭스

In [ ]:
# RFM 세그먼트와 이탈 위험을 결합한 Action Matrix
# (Online Retail RFM 세그먼트 × Telco 이탈 예측 프레임워크)

matrix_data = pd.DataFrame({
    'Segment': ['Champions', 'Loyal Customers', 'At-Risk', 'Lost'],
    'Churn Risk': ['Low', 'Low-Medium', 'High', 'Very High'],
    'Value': ['High', 'Medium', 'High (was)', 'Low'],
    'Priority': [3, 2, 1, 4],  # 1 = highest priority
    'Action': [
        'VIP 유지: 얼리 액세스, 로열티 보상',
        '업셀: 카테고리 확장, 번들 제안',
        '긴급 리텐션: 개인화 할인, 1:1 연락',
        '저비용 리타겟팅 또는 자연 이탈 수용'
    ],
    'Expected ROI': ['Maintenance', 'Growth', 'Highest ROI', 'Lowest ROI']
})

print('=== Segment × Churn Risk Action Matrix ===')
matrix_data.set_index('Segment')

In [ ]:
# Action Matrix 시각화 (2×2 grid)
fig, ax = plt.subplots(figsize=(10, 8))

# 2×2 매트릭스: X=이탈 위험, Y=고객 가치
positions = {
    'Champions': (0.25, 0.75),      # Low risk, High value
    'Loyal Customers': (0.25, 0.25), # Low risk, Medium value
    'At-Risk': (0.75, 0.75),         # High risk, High value
    'Lost': (0.75, 0.25),            # High risk, Low value
}

colors_seg = {'Champions': '#59a14f', 'Loyal Customers': '#4e79a7',
              'At-Risk': '#f28e2b', 'Lost': '#e15759'}

sizes = {'Champions': 3000, 'Loyal Customers': 2000, 'At-Risk': 3500, 'Lost': 1500}

for seg, (x, y) in positions.items():
    ax.scatter(x, y, s=sizes[seg], c=colors_seg[seg], alpha=0.6, edgecolors='white', linewidth=2)
    ax.annotate(seg, (x, y), fontsize=12, fontweight='bold', ha='center', va='center')

# 축 설정
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('Churn Risk →', fontsize=13, fontweight='bold')
ax.set_ylabel('Customer Value →', fontsize=13, fontweight='bold')
ax.set_title('Customer Segment × Churn Risk Matrix', fontsize=15, fontweight='bold')

# 사분면 라벨
ax.text(0.25, 0.98, 'MAINTAIN', ha='center', fontsize=10, color='gray', style='italic')
ax.text(0.75, 0.98, 'ACT NOW', ha='center', fontsize=10, color='#e15759', fontweight='bold', style='italic')
ax.text(0.25, 0.02, 'DEVELOP', ha='center', fontsize=10, color='gray', style='italic')
ax.text(0.75, 0.02, 'DEPRIORITIZE', ha='center', fontsize=10, color='gray', style='italic')

# 경계선
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.3)

ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()

## 5. 통합 액션 플랜

| 우선순위 | 액션 | 대상 | 예상 임팩트 | 근거 |
|----------|------|------|------------|------|
| **P0** | Month-to-month → 연간 계약 유도 | 이탈 위험 상위 20% | 이탈률 42% → ~10% | Contract이 #1 이탈 예측 변수 (SHAP) |
| **P1** | 초기 3개월 온보딩 강화 | tenure < 6개월 | 초기 이탈 30%↓ | tenure-churn 음의 상관 (Cohen's d Large) |
| **P1** | At-Risk 세그먼트 긴급 리텐션 | RFM At-Risk + 모델 예측 이탈 | ROI 최대 구간 | 과거 고가치 + 현재 이탈 위험 |
| **P2** | 부가서비스 번들 제안 | OnlineSecurity/TechSupport 미가입 | 이탈률 15%↓ | 부가서비스 미가입 시 이탈률 2배 |
| **P3** | 요금 민감도 세그먼트 할인 | MonthlyCharges 상위 25% + 이탈 예측 | 가격 불만 해소 | MonthlyCharges Cohen's d Medium |

In [ ]:
# 임팩트 요약 대시보드
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: 이탈 비용 요약
kpi_labels = ['Monthly\nRevenue\nat Risk', 'Annual\nRevenue\nat Risk', 'Net Benefit\n(Retention\nCampaign)']
kpi_values = [monthly_revenue_at_risk, annual_revenue_at_risk, net_benefit]
kpi_colors = ['#e15759', '#e15759', '#59a14f']

bars = axes[0].bar(kpi_labels, kpi_values, color=kpi_colors, width=0.5)
for bar, v in zip(bars, kpi_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Financial Impact Summary', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Amount ($)')

# Panel 2: 모델 성능 요약
model_metrics = ['Recall\n(Detection)', 'Precision\n(Accuracy)', 'ROC-AUC\n(Overall)']
# 보수적 추정치 (실제 값은 04_churn_modeling에서 결정)
model_values = [0.80, 0.65, 0.85]
bars = axes[1].bar(model_metrics, model_values, color=['#4e79a7', '#f28e2b', '#59a14f'], width=0.5)
for bar, v in zip(bars, model_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{v:.0%}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Model Performance Targets', fontsize=13, fontweight='bold')

# Panel 3: 세그먼트별 Action Priority
segments_list = ['At-Risk\n(P0)', 'Tenure<6mo\n(P1)', 'No Add-ons\n(P2)', 'High Price\n(P3)']
impact_est = [40, 30, 15, 10]  # estimated % impact
colors_p = ['#e15759', '#f28e2b', '#4e79a7', '#76b7b2']
bars = axes[2].barh(segments_list[::-1], impact_est[::-1], color=colors_p[::-1])
for bar, v in zip(bars, impact_est[::-1]):
    axes[2].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{v}%', va='center', fontsize=10, fontweight='bold')
axes[2].set_xlabel('Estimated Churn Reduction (%)')
axes[2].set_title('Action Priority & Impact', fontsize=13, fontweight='bold')

plt.suptitle('Churn Prevention — Business Impact Dashboard', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. 포트폴리오 3개 축 연결

이 ML 분석 프로젝트는 포트폴리오의 세 번째 축으로, 기존 두 프로젝트와 상호 보완합니다:

| 프로젝트 | 역할 | 핵심 역량 |
|----------|------|----------|
| **FRE (SaaS)** | 분석 프레임워크 제품화 | TypeScript 엔진 설계, 통계 검정 구현, React SaaS 개발 |
| **SQL Analysis** | 실데이터 인사이트 도출 | BigQuery SQL, 퍼널/리텐션/세그먼트 분석 |
| **ML Analysis** | 예측 모델링 + 비즈니스 임팩트 | scikit-learn, RFM 세그멘테이션, 이탈 예측, ROI 계산 |

### 역량 연결 관계:

```
┌─────────────┐     ┌──────────────┐     ┌──────────────┐
│ FRE (SaaS)  │     │ SQL Analysis │     │ ML Analysis  │
│             │     │              │     │              │
│ 지표 설계    │────→│ 실데이터 분석 │────→│ 예측 모델링   │
│ 엔진 구현    │     │ SQL 쿼리     │     │ Feature Eng. │
│ SaaS 제품화  │     │ 통계 검정    │     │ SHAP 해석    │
│             │     │ 시각화       │     │ ROI 계산     │
└─────────────┘     └──────────────┘     └──────────────┘
     도구 만들기          도구 활용          의사결정 지원
```

### Key Message:
**"분석 지표를 설계하고(FRE), 실데이터에서 인사이트를 도출하고(SQL), ML로 비즈니스 임팩트를 정량화할 수 있습니다."**